In [ ]:
import datetime
import time
import threading
import random
import rclpy

from IPython.display import clear_output
from rclpy.node import Node
from rclpy.action import ActionClient
from rclpy.executors import MultiThreadedExecutor
from geographic_msgs.msg import GeoPoint
from lotusim_msgs.msg import VesselPositionArray
from lotusim_msgs.msg import MASCmd as MASCmdMsg
from lotusim_msgs.action import MASCmd, MASCmdArray
from lotusim_msgs.srv import SetWaypoints

This notebook demonstrates how to spawn different types of vessels in LOTUSim:
- **LRAUV** (underwater): physics-driven via XDyn WebSocket
- **x500** (aerial drone): physics-driven via ROS2
- **DTMB hull** (surface): waypoint-following, single or multiple

Each spawn method returns a future from the action server. Since the ROS executor runs in a background thread, we poll `.done()` instead of using `rclpy.spin_until_future_complete`.

In [ ]:
SPAWN_LATITUDE = 1.2605794416293148
SPAWN_LONGITUDE = 103.7516212463379
SPAWN_ALTITUDE = 0.0
OFFSET = 0.0001

class ExampleNode(Node):
    def __init__(self):
        super().__init__('example_spawn_node', namespace='/lotusim')

        self.pose_subscription = self.create_subscription(
            VesselPositionArray,
            '/lotusim/poses',
            self.poses_callback,
            10)
        self.mas_action_client = ActionClient(
            self,
            MASCmd,
            '/lotusim/mas_cmd'
        )
        self.mas_array_action_client = ActionClient(
            self,
            MASCmdArray,
            '/lotusim/mas_cmd_array'
        )
        self.vessel_poses = {}
        self.spawned_vessels = []  # track spawned vessels for cleanup
        self.waypoint_client = {}
        self.vessel_id = 0  # id number for the number of model in the simulation. Used in randomizing location spawned

    def poses_callback(self, msg):
        for vessel_position in msg.vessels:
            lat = vessel_position.geo_point.latitude
            lon = vessel_position.geo_point.longitude
            alt = vessel_position.geo_point.altitude
            self.vessel_poses[vessel_position.vessel_name] = (lat, lon, alt)

    def delete_all_vessels(self):
        if not self.spawned_vessels:
            print("No vessels to delete")
            return

        if not self.mas_array_action_client.wait_for_server(timeout_sec=2.0):
            print("Action server not available for cleanup")
            return

        goal_msg = MASCmdArray.Goal()
        for name in self.spawned_vessels:
            msg = MASCmdMsg()
            msg.cmd_type = MASCmdMsg.DELETE_CMD
            msg.vessel_name = name
            goal_msg.cmd.append(msg)

        # send goal and wait -> executor in background thread processes it
        goal_future = self.mas_array_action_client.send_goal_async(goal_msg)
        while not goal_future.done():
            time.sleep(0.1)

        goal_handle = goal_future.result()
        if not goal_handle or not goal_handle.accepted:
            print("Delete goal rejected")
            return

        result_future = goal_handle.get_result_async()
        while not result_future.done():
            time.sleep(0.1)

        print("All vessels deleted")
        self.spawned_vessels.clear()

    def spawn_ship_with_dynamics(self):
        msg = MASCmdMsg()
        msg.cmd_type = MASCmdMsg.CREATE_CMD
        msg.model_name = "lrauv"
        vessel_name = f"lrauv_{self.vessel_id}"
        msg.vessel_name = vessel_name

        geo = GeoPoint()
        geo.latitude = SPAWN_LATITUDE + OFFSET * self.vessel_id * random.choice([-1, 1])
        geo.longitude = SPAWN_LONGITUDE + OFFSET * self.vessel_id * random.choice([-1, 1])
        geo.altitude = SPAWN_ALTITUDE
        msg.geo_point = geo

        msg.sdf_string = """
        <lotus_param>
            <render_interface>
                <publish_render>true</publish_render>
                <renderer_type_name>lrauv</renderer_type_name>
            </render_interface>
            <physics_engine_interface>
            <underwater>
                <interface_type>XDynWebSocket</interface_type>
                <uri>ws://127.0.0.1:12346</uri>
                <thrusters>
                    <thrusters1>thruster1</thrusters1>
                </thrusters>
            </underwater>
            <surface>
                <interface_type>XDynWebSocket</interface_type>
                <uri>ws://127.0.0.1:12345</uri>
                <thrusters>
                    <thrusters1>thruster1</thrusters1>
                </thrusters>
            </surface>
            <init_state>Underwater</init_state>
            </physics_engine_interface>
        </lotus_param>
        """
        self.vessel_id += 1
        self.spawned_vessels.append(vessel_name)
        goal_msg = MASCmd.Goal()
        goal_msg.cmd = msg
        self.mas_action_client.wait_for_server()
        return self.mas_action_client.send_goal_async(goal_msg)

    def spawn_aerial_drone(self):
        msg = MASCmdMsg()
        msg.cmd_type = MASCmdMsg.CREATE_CMD
        msg.model_name = "x500"
        vessel_name = f"x500_{self.vessel_id}"
        msg.vessel_name = vessel_name

        geo = GeoPoint()
        geo.latitude = SPAWN_LATITUDE + OFFSET * self.vessel_id * random.choice([-1, 1])
        geo.longitude = SPAWN_LONGITUDE + OFFSET * self.vessel_id * random.choice([-1, 1])
        geo.altitude = 10.0
        msg.geo_point = geo

        msg.sdf_string = """
        <lotus_param>
            <render_interface>
                <publish_render>true</publish_render>
                <renderer_type_name>lrauv</renderer_type_name>
            </render_interface>
            <physics_engine_interface>
                  <aerial>
                    <interface_type>ROS2</interface_type>
                    <namespace>aerialWorld</namespace>
                  </aerial>
                  <init_state>Aerial</init_state>
            </physics_engine_interface>
        </lotus_param>
        """
        self.vessel_id += 1
        self.spawned_vessels.append(vessel_name)
        goal_msg = MASCmd.Goal()
        goal_msg.cmd = msg
        self.mas_action_client.wait_for_server()
        return self.mas_action_client.send_goal_async(goal_msg)

    def spawn_circling_ship(self):
        msg = MASCmdMsg()
        msg.cmd_type = MASCmdMsg.CREATE_CMD
        msg.model_name = "dtmb_hull"
        vessel_name = f"dtmb_{self.vessel_id}"
        msg.vessel_name = vessel_name

        geo = GeoPoint()
        geo.latitude = SPAWN_LATITUDE + OFFSET * self.vessel_id * random.choice([-1, 1])
        geo.longitude = SPAWN_LONGITUDE + OFFSET * self.vessel_id * random.choice([-1, 1])
        geo.altitude = SPAWN_ALTITUDE
        msg.geo_point = geo

        msg.sdf_string = """
        <lotus_param>
            <waypoint_follower>
                <follower>
                    <loop>true</loop>
                    <linear_accel_limit>0.5</linear_accel_limit>
                    <angular_accel_limit>0.005</angular_accel_limit>
                    <angular_velocities_limits>0.01</angular_velocities_limits>
                    <range_tolerance>2</range_tolerance>
                    <circle>
                        <radius>10</radius>
                    </circle>
                </follower>
            </waypoint_follower>
        </lotus_param>
        """
        self.vessel_id += 1
        self.spawned_vessels.append(vessel_name)
        goal_msg = MASCmd.Goal()
        goal_msg.cmd = msg
        self.mas_action_client.wait_for_server()
        return self.mas_action_client.send_goal_async(goal_msg)

    def spawn_multiple_circling_ship(self, number_of_ships):
        goal_msg = MASCmdArray.Goal()

        for i in range(number_of_ships):
            msg = MASCmdMsg()
            msg.cmd_type = MASCmdMsg.CREATE_CMD
            msg.model_name = "dtmb_hull"
            vessel_name = f"dtmb_{self.vessel_id}"
            msg.vessel_name = vessel_name

            geo = GeoPoint()
            geo.latitude = SPAWN_LATITUDE + OFFSET * self.vessel_id * random.choice([-1, 1])
            geo.longitude = SPAWN_LONGITUDE + OFFSET * self.vessel_id * random.choice([-1, 1])
            geo.altitude = SPAWN_ALTITUDE
            msg.geo_point = geo

            msg.sdf_string = """
            <lotus_param>
                <waypoint_follower>
                    <follower>
                        <loop>true</loop>
                        <linear_accel_limit>0.5</linear_accel_limit>
                        <angular_accel_limit>0.005</angular_accel_limit>
                        <angular_velocities_limits>0.01</angular_velocities_limits>
                        <range_tolerance>2</range_tolerance>
                        <circle>
                            <radius>100</radius>
                        </circle>
                    </follower>
                </waypoint_follower>
            </lotus_param>
            """
            goal_msg.cmd.append(msg)
            self.vessel_id += 1
            self.spawned_vessels.append(vessel_name)

        self.mas_array_action_client.wait_for_server()
        return self.mas_array_action_client.send_goal_async(goal_msg)

    def setupRosForModel(self, model_name: str):
        if model_name not in self.waypoint_client:
            client = self.create_client(SetWaypoints, model_name + '/waypoints')
            for attempt in range(1, 4):
                if client.wait_for_service(timeout_sec=1.0):
                    self.waypoint_client[model_name] = client
                    return
                print(f'Waiting for waypoint service... (attempt {attempt}/3)')
            print(f'Failed to connect to waypoint service for {model_name}')

    def send_random_waypoint_request(self, model_name: str):
        if model_name not in self.waypoint_client:
            try:
                self.setupRosForModel(model_name)
            except ValueError:
                print("Sending waypoint failed. Try again.")
                return
        request = SetWaypoints.Request()

        latitude = SPAWN_LATITUDE + OFFSET * self.vessel_id * random.choice([-1, 1])
        longitude = SPAWN_LONGITUDE + OFFSET * self.vessel_id * random.choice([-1, 1])
        request.path = [GeoPoint(latitude=latitude, longitude=longitude, altitude=SPAWN_ALTITUDE)]
        request.loop = False

        future = self.waypoint_client[model_name].call_async(request)
        while not future.done():
            time.sleep(0.1)

        response = future.result()
        if response.success:
            print(f'Waypoint set: ({latitude:.7f}, {longitude:.7f})')
        else:
            print('Failed to set waypoints')

In [ ]:
if not rclpy.ok():
    rclpy.init(args=None)

try:
    stop_executor()
except NameError:
    pass
except Exception as e:
    print(f"Could not destroy resources: {e}")

node = ExampleNode()
executor = MultiThreadedExecutor()
executor.add_node(node)
stop_event = threading.Event()

def spin_executor():
    executor.spin()

def stop_executor():
    """Call this function to stop the spinning thread"""
    try:
        node.delete_all_vessels()   # cleanup before stopping
        stop_event.set()
        executor.shutdown()
        spin_thread.join()
        display_thread.join()
        node.destroy_node()
        time.sleep(2)
        print("Executor stopped successfully")
    except Exception as e:
        print(f"Error stopping executor: {e}")

def display_loop():
    while not stop_event.is_set():
        clear_output(wait=True)
        print(f"[{datetime.datetime.now().strftime('%H:%M:%S')}]")
        for name, (lat, lon, alt) in node.vessel_poses.items():
            print(f"  {name}: lat={lat:.6f}, lon={lon:.6f}, alt={alt:.2f}")
        time.sleep(1)

display_thread = threading.Thread(target=display_loop, daemon=True)
display_thread.start()
spin_thread = threading.Thread(target=spin_executor, daemon=True)
spin_thread.start()

### Spawn vessels

Run any of the cells below to spawn the desired vessel type. Each cell polls the action future until it resolves, then prints the spawned vessel name.

In [ ]:
# Spawn multiple circling DTMB surface ships
future = node.spawn_multiple_circling_ship(2)
while not future.done():
    time.sleep(0.1)
goal_handle = future.result()
result_future = goal_handle.get_result_async()
while not result_future.done():
    time.sleep(0.1)
print("Spawned multiple circling ships")

In [ ]:
# Spawn a single circling DTMB surface ship
future = node.spawn_circling_ship()
while not future.done():
    time.sleep(0.1)
goal_handle = future.result()
result_future = goal_handle.get_result_async()
while not result_future.done():
    time.sleep(0.1)
vessel_name = result_future.result().result.name
print(f"Spawned circling ship: {vessel_name}")

In [ ]:
# Spawn an LRAUV with XDyn physics
future = node.spawn_ship_with_dynamics()
while not future.done():
    time.sleep(0.1)
goal_handle = future.result()
result_future = goal_handle.get_result_async()
while not result_future.done():
    time.sleep(0.1)
vessel_name = result_future.result().result.name
print(f"Spawned underwater vessel: {vessel_name}")

In [ ]:
# Spawn an x500 aerial drone
future = node.spawn_aerial_drone()
while not future.done():
    time.sleep(0.1)
goal_handle = future.result()
result_future = goal_handle.get_result_async()
while not result_future.done():
    time.sleep(0.1)
vessel_name = result_future.result().result.name
print(f"Spawned aerial drone: {vessel_name}")

In [ ]:
# Send a random waypoint to a vessel (e.g. dtmb_0)
node.send_random_waypoint_request("dtmb_0")

In [ ]:
stop_executor()